In [23]:
from IPython.display import display, Markdown, HTML
import os
import pyreadstat
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import missingno as msno
import pyarrow
from pathlib import Path

In [24]:
class Loader:
    """
    Encapsulates file ingestion and cohort verification.
    """
    
    def __init__(self, data_dir: str = "../outputs"):
        self.data_dir = Path(data_dir)

    def base(self, filename: str = "nhanes_processed_cohort.parquet") -> pd.DataFrame:
        path = self.data_dir / filename
        print(f"[DataLoader] Ingesting registry from: {path}")
        
        df = pd.read_parquet(path)
        
        # Immediate analytical check to guarantee no data degradation
        print(f"[DataLoader] Verification Successful. N = {df.shape[0]} rows.")
        return df

In [ ]:
class Calculator:

    def __init__(self, dataframe):
        self.df = dataframe.copy()

    def table_one(self):
        design_cols = ["groups", "WTSAF2YR"]
        target_vars = ["sys_bp", "dia_bp", "LBXGLU", "LBXTLG", "LBDHDD", "LUXSMED", "ckd_epi"]

        self.df[design_cols] = self.df[design_cols].apply(pd.to_numeric, errors='coerce')
        self.df[target_vars] = self.df[target_vars].apply(pd.to_numeric, errors='coerce')

        # Global Clean: Filter to cohort and drop records missing crucial survey design/weight info
        cohort_df = self.df[self.df["cohort"] == True].dropna(subset=design_cols).copy()

        # Map friendly names to their respective boolean columns
        # We skip 'phenotype_outliers' entirely
        phenotype_map = {
            "control": "phenotype_control",
            "visceral": "phenotype_visceral",
            "classic": "phenotype_obesity"
        }

        results = []

        # Outer Loop: Iterate through each phenotype mapping
        for pheno_name, col_name in phenotype_map.items():
            
            # Subset to patients who strictly have 'True' in the respective boolean column
            pheno_df = cohort_df[cohort_df[col_name] == True]
            
            # Inner Loop: Analyze each target variable using pairwise deletion
            for var in target_vars:
                
                # Local Clean: Only keep non-null values for the variable we are current measuring
                local_df = pheno_df.dropna(subset=[var]).copy()
                
                # Guard clause if a subpopulation is empty for a specific rare measure
                if len(local_df) == 0:
                    continue
                    
                # Create constant and force index alignment
                exog = pd.DataFrame({'const': 1.0}, index=local_df.index)
                
                # Fit WLS model
                model = sm.WLS(
                    local_df[var], 
                    exog, 
                    weights=local_df["WTSAF2YR"]
                )
                
                # Standard errors clustered by MVU
                res = model.fit(
                    cov_type='cluster', 
                    cov_kwds={'groups': local_df["groups"]}
                )
                
                # Collect statistics
                results.append({
                    "Phenotype": pheno_name,
                    "Variable": var,
                    "N_Valid": len(local_df),
                    "Mean": res.params.iloc[0],
                    "SE": res.bse.iloc[0],
                    "LCI": res.conf_int().iloc[0, 0],
                    "UCI": res.conf_int().iloc[0, 1]
                })

        # Compile into a clean stratified Table One DataFrame
        table_one_stratified = pd.DataFrame(results)

        # Pivot the table for a side-by-side comparative layout
        pivot_table = table_one_stratified.pivot(
            index="Variable", 
            columns="Phenotype", 
            values=["Mean", "SE", "N_Valid"]
        )

        # Reindex the column so it follows a pathophysiological progression
        pivot_table = pivot_table.reindex(
            columns=['control', 'visceral', 'classic'], 
            level='Phenotype'
        )

        return display(pivot_table)

In [26]:
loader = Loader(data_dir="../outputs")
df = loader.base()

[DataLoader] Ingesting registry from: ..\outputs\nhanes_processed_cohort.parquet
[DataLoader] Verification Successful. N = 11933 rows.


In [27]:
Table_one = Calculator(df).table_one()

Mean                                SE                      \
Phenotype     control    visceral     classic   control  visceral   classic   
Variable                                                                      
LBDHDD      60.697991   53.494724   49.593355  0.926117  0.799929  0.592825   
LBXGLU      98.717548  103.937462  111.306508  4.190328  1.212153  2.020791   
LBXTLG      74.164913  117.455000  124.362528  2.785569  3.004310  2.746552   
LUXSMED      4.856355    5.417103    6.794321  0.120195  0.368683  0.227098   
ckd_epi    108.142679   99.571271  101.407746  0.983028  0.705692  0.955740   
dia_bp      68.587021   74.113076   78.644908  0.577680  0.468989  0.457153   
sys_bp     112.746366  118.292996  119.305309  0.735249  0.737315  0.457317   

          N_Valid                   
Phenotype control visceral classic  
Variable                            
LBDHDD      290.0    630.0   787.0  
LBXGLU      290.0    630.0   787.0  
LBXTLG      290.0    630.0   787.0  
LUXSMED     288.0    613.0   750.0  
ckd_epi     290.0    630.0   787.0  
dia_bp      288.0    623.0   766.0  
sys_bp      288.0    623.0   766.0